# TicTacToe — Benchmark de funciones de evaluación

Comparación entre `TicTacToe.eval` (líneas abiertas) y `TicTacToeEval.eval` (amenazas ponderadas)
en partidas de Minimax vs Minimax a profundidad variable.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from games.tictactoe.tictactoe import TicTacToe, TicTacToeEval
from agents.minimax import MiniMax

FIGURES_DIR = os.path.join('..', 'informe', 'figures', 'TicTacToe')
os.makedirs(FIGURES_DIR, exist_ok=True)

def savefig(fig, name):
    path = os.path.join(FIGURES_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f'Figura guardada en {path}')

def sync(src, dst):
    dst.env.board.squares = src.env.board.squares.copy()
    dst.env.rewards       = src.env.rewards.copy()
    dst.env.terminations  = src.env.terminations.copy()
    dst.env.truncations   = src.env.truncations.copy()
    dst.env.agent_selection = src.env.agent_selection
    dst._update()

def play_match(cls0, cls1, depth, N=20, seed=42):
    np.random.seed(seed)
    same = cls0 is cls1

    g0 = cls0()
    g1 = g0 if same else cls1()

    m0 = MiniMax(game=g0, agent=g0.agents[0], depth=depth)
    m1 = MiniMax(game=g1, agent=g1.agents[1], depth=depth)

    results = []
    for _ in range(N):
        g0.reset()
        if not same:
            sync(g0, g1)

        t0 = time.perf_counter()
        while not g0.game_over():
            if g0.agent_selection == g0.agents[0]:
                a = m0.action()
            else:
                if not same:
                    sync(g0, g1)
                a = m1.action()
            g0.step(a)
            if not same:
                sync(g0, g1)
        elapsed = time.perf_counter() - t0

        r = g0.rewards[g0.agents[0]]
        results.append(dict(
            outcome='W' if r > 0 else ('L' if r < 0 else 'D'),
            reward_0=r,
            time=elapsed,
        ))
    return results

def summarize(results, label):
    outs  = [r['outcome'] for r in results]
    times = [r['time'] for r in results]
    return dict(
        config=label,
        W=outs.count('W'),
        D=outs.count('D'),
        L=outs.count('L'),
        avg_time_s=round(np.mean(times), 3),
    )

## Benchmark a profundidad 9

Con `depth=9` el árbol siempre llega a nodos terminales — la función `eval` no se invoca.
Ambas heurísticas producen juego óptimo; el resultado esperado son empates.

In [ ]:
DEPTH = 9
N = 5

configs = [
    ('TicTacToe vs TicTacToe',         TicTacToe,     TicTacToe),
    ('TicTacToeEval vs TicTacToeEval', TicTacToeEval, TicTacToeEval),
]

rows_d9 = []
for label, cls0, cls1 in configs:
    print(f'Jugando {label} (depth={DEPTH}, N={N})...')
    res = play_match(cls0, cls1, depth=DEPTH, N=N)
    rows_d9.append(summarize(res, label))

df_d9 = pd.DataFrame(rows_d9)
print()
print(df_d9.to_string(index=False))

## Benchmark a profundidades variables

A profundidades bajas `eval` sí se invoca — aquí se diferencian las dos heurísticas.

In [ ]:
depths = [1, 2, 3, 5]
N_VAR = 20
rows_var = []

for d in depths:
    for label, cls0, cls1 in configs:
        res = play_match(cls0, cls1, depth=d, N=N_VAR)
        s = summarize(res, label)
        s['depth'] = d
        rows_var.append(s)
    print(f'depth={d} listo')

df_var = pd.DataFrame(rows_var)
df_var.pivot_table(index='depth', columns='config', values=['W', 'D', 'L'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
styles = {
    'TicTacToe vs TicTacToe':         ('tab:blue',   'o', '-'),
    'TicTacToeEval vs TicTacToeEval': ('tab:orange', 's', '-'),
}

for cfg, (color, marker, ls) in styles.items():
    sub = df_var[df_var.config == cfg]
    axes[0].plot(sub.depth, sub.W, color=color, marker=marker, ls=ls, label=cfg)
    axes[1].plot(sub.depth, sub.avg_time_s, color=color, marker=marker, ls=ls, label=cfg)

axes[0].set_xlabel('Profundidad Minimax')
axes[0].set_ylabel('Victorias de agent_0 (de 30)')
axes[0].set_title('Victorias por eval y profundidad')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].set_xlabel('Profundidad Minimax')
axes[1].set_ylabel('Tiempo promedio por partida (s)')
axes[1].set_title('Tiempo de cómputo')
axes[1].set_yscale('log')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

fig.tight_layout()
savefig(fig, 'eval_benchmark.png')
plt.show()